# Tutorial 14 - From an AI Situation to a Working CNN
## A beginner-first integrated lab using the UCU Smart Farm story

This notebook is designed for students who **recognize AI concepts when they see the answer, but find it difficult to generate the answer from a new situation**.

The central skill is therefore not “memorize the model.” It is:

> **Read the situation → identify the goal → identify the input and target → determine the form of the required output → justify the AI/ML task → only then choose a method.**

We will repeatedly ask **why** an answer is correct and **why plausible alternatives are not correct**.

### Main story
A teaching prototype receives a cropped photograph of one leaf and should help a farmer interpret the visible condition of the leaf.

The notebook is completely self-contained: it generates its own small image dataset, trains a small CNN, evaluates it, and includes transfer exercises from other domains.

## How to use this notebook

1. Read each scenario **before** opening the answer.
2. Write your own reasoning in the exercise cell or on paper.
3. Do not identify a task from the *type of data alone*. An image can support classification, regression, clustering, segmentation, or detection depending on the question.
4. Run the code cells in order in Google Colab.
5. At the end, you should be able to explain a new AI problem without copying the wording of an earlier example.

## Learning outcomes

By the end of the tutorial, you should be able to:

- distinguish classification, regression, clustering, object detection, segmentation, and agent-based decision making;
- identify the **observation/prediction unit**, **features/input**, and **target/output** in a new scenario;
- justify why a problem belongs to one task type and why it does **not** belong to plausible alternatives;
- explain the relationship between AI, machine learning, deep learning, computer vision, and CNNs;
- explain what convolution, ReLU, pooling, logits, softmax, loss, and an optimizer do;
- train and evaluate a small image classifier;
- interpret accuracy, precision, recall, and a confusion matrix;
- distinguish a predictive model from a complete intelligent agent;
- transfer the same reasoning procedure to unfamiliar situations.

# Part 1 - Diagnose the problem before touching the model

A common mistake is to see a photograph and immediately say **“CNN”** or **“classification.”**

That reverses the correct reasoning order.

The first question should be:

> **What exactly do we want the system to output?**

## Scenario A - One photograph, one condition label

A farmer photographs one leaf. The system must output one of these predefined labels:

- `healthy-looking`
- `spot-pattern`
- `yellowing-pattern`

Before revealing the answer, compare the possible task types.

| Possible task | What the answer would look like | Does it match this scenario? |
|---|---|---|
| Classification | one predefined category | ? |
| Regression | a numerical quantity | ? |
| Clustering | groups discovered without supplied target labels | ? |
| Object detection | category **and location** of one or more objects | ? |
| Segmentation | a label for each relevant pixel/region | ? |

**Your reasoning:** What is the required output, and what does that imply?

In [ ]:
my_reasoning_A = {
    "prediction_unit": "",
    "input": "",
    "target": "",
    "form_of_output": "",
    "task_type": "",
    "why": "",
    "why_not_regression": "",
    "why_not_clustering": ""
}
my_reasoning_A

<details>
<summary><b>Reveal the reasoning for Scenario A</b></summary>

### Answer: supervised multiclass classification

The required output is **one member of a predefined set of categories**:


a. healthy-looking  
b. spot-pattern  
c. yellowing-pattern

That makes the task **classification**.

It is **multiclass** because there are more than two possible classes.

It is **supervised** because, during training, example images are paired with the correct class labels.

### Why not regression?
Regression would require a numerical target, for example:

- affected leaf area = `27.4%`, or
- estimated chlorophyll index = `41.8`.

### Why not clustering?
Clustering would be appropriate if the images had **no predefined target labels** and we asked the algorithm to discover groups of visually similar leaves.

### Important lesson
The task is not classification merely because it uses images. It is classification because of the **form of the required output**.

</details>

## The same image can support different AI/ML problems

Suppose the input is always a leaf photograph. Changing only the question changes the task.

| Question asked of the image | Required output | Task |
|---|---|---|
| “Which known condition is visible?” | `healthy`, `spot`, or `yellowing` | Classification |
| “What percentage of the leaf is affected?” | e.g. `31.6%` | Regression |
| “What natural visual groups exist in these unlabeled images?” | discovered groups | Clustering |
| “Where are all visible lesions?” | boxes/locations + classes | Object detection |
| “Which exact pixels belong to lesions?” | pixel-level mask | Segmentation |

### Rule to remember

> **Do not identify the task from the input data type. Identify it from the goal and the required output.**

In [ ]:
# A simple reasoning diagram drawn inside the notebook.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis('off')
labels = [
    "1. Situation\nWhat is happening?",
    "2. Goal\nWhat must the system do?",
    "3. Input & target\nWhat is known? What is required?",
    "4. Output form\nCategory, number, groups, pixels...?",
    "5. Task type\nClassify, regress, cluster...?"
]
xs = [0.02, 0.21, 0.40, 0.61, 0.82]
width = 0.16
for x, lab in zip(xs, labels):
    box = FancyBboxPatch((x,0.35), width,0.34, boxstyle="round,pad=0.02", linewidth=1.5, facecolor='white')
    ax.add_patch(box)
    ax.text(x+width/2,0.52,lab,ha='center',va='center',fontsize=10)
for i in range(len(xs)-1):
    ax.add_patch(FancyArrowPatch((xs[i]+width,0.52),(xs[i+1],0.52),arrowstyle='->',mutation_scale=15,linewidth=1.5))
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Reasoning order: diagnose before choosing a model", fontsize=14)
plt.show()

## A reusable four-question test

For almost every beginner ML scenario, ask:

1. **What is one observation or prediction unit?**  
   One student? One transaction? One image? One house? One hour?

2. **What information is available to the system?**  
   These become candidate **features/inputs**.

3. **What should the system produce?**  
   This is the **target/output**, if a known target exists.

4. **What form does the answer take?**

A useful first mapping is:

- predefined category → **classification**
- numerical quantity → **regression**
- no supplied target; discover groups → **clustering**
- object categories + positions → **object detection**
- pixel/region labels → **segmentation**
- action chosen through interaction with an environment → **agent / possibly reinforcement-learning framing**

## Worked example - student support

**Situation:** A university wants to flag students who may require additional academic support by Week 5.

Possible inputs include attendance, quiz scores, missed assignments, and LMS activity.

Required output:

> `needs-support` or `not-currently-flagged`

### Reasoning

- prediction unit: **one student at Week 5**;
- inputs/features: Week-5 information available at that time;
- target: whether the student belongs to one of two predefined categories;
- output form: **category**;
- therefore: **binary classification**.

If the question changed to **“predict the student's final numerical mark”**, the same general domain would become a **regression** problem.

If the question changed to **“discover common student-engagement patterns without predefined labels”**, it would become a **clustering** problem.

## Mixed practice - classify the *question*, not the domain

For each scenario, identify the prediction unit, target/output, and task type.

1. Predict whether a mobile-money transaction is `fraud` or `legitimate`.
2. Predict tomorrow's electricity demand in kWh.
3. Group customers by usage behaviour when no group labels are supplied.
4. Find and locate all vehicles in a road-camera image.
5. Mark the exact pixels occupied by a tumour in a medical image.
6. Predict whether an email is spam.
7. Predict rainfall amount in millimetres.

In [ ]:
my_mixed_answers = {
    1: {"task":"", "why":""},
    2: {"task":"", "why":""},
    3: {"task":"", "why":""},
    4: {"task":"", "why":""},
    5: {"task":"", "why":""},
    6: {"task":"", "why":""},
    7: {"task":"", "why":""},
}
my_mixed_answers

<details>
<summary><b>Reveal the mixed-practice answers</b></summary>

1. **Classification** - the output is one of two predefined categories: fraud or legitimate.  
2. **Regression** - electricity demand is a numerical quantity.  
3. **Clustering** - no target group labels are supplied; the goal is to discover groups.  
4. **Object detection** - the system must identify vehicle categories and their locations.  
5. **Segmentation** - the required answer is pixel-level.  
6. **Classification** - spam/not-spam is a predefined categorical target.  
7. **Regression** - rainfall amount is numerical.

Notice that tasks 4 and 5 both use images, but they are not ordinary image classification because their required outputs are different.

</details>

# Part 2 - Observation, features, target, metadata, identifiers, and leakage

Correct task framing requires more than naming “classification.” We must define exactly what one example represents and what information the model is allowed to use.

## Key terms

### Observation / prediction unit
One item about which the system makes a prediction.

Examples: one transaction, one student, one image, one patient visit, one hour.

### Feature / input
Information available to the model **at prediction time**.

### Target / label
The correct answer we want a supervised model to learn to predict.

### Metadata
Descriptive information about the observation. Metadata may or may not be useful as a feature.

### Identifier
A value whose main purpose is to uniquely identify an item, such as `case_id`. It is usually not a meaningful predictive feature.

### Leakage
Information that would not legitimately be available when the real prediction is made, or that directly/indirectly reveals the answer.

> A model can look excellent during training and still be invalid if it learned from leaked information.

## Apply the terms to the smart-farm story

Suppose a table contains:

- `case_id`
- `current_image`
- `humidity_percent`
- `field_section`
- `visual_pattern` - expert label
- `lab_result_received_two_days_later`

For the task **“predict the visible pattern immediately from the current photograph”**, decide the role of each variable.

In [ ]:
my_column_roles = {
    "case_id": "",
    "current_image": "",
    "humidity_percent": "",
    "field_section": "",
    "visual_pattern": "",
    "lab_result_received_two_days_later": ""
}
my_column_roles

<details>
<summary><b>Reveal a defensible answer</b></summary>

- `case_id`: **identifier**; normally excluded from predictive features.
- `current_image`: main **feature/input**.
- `humidity_percent`: possible contextual feature if it is truly available at prediction time and the task design permits it.
- `field_section`: metadata; it could be used, but doing so requires care because location can create spurious shortcuts.
- `visual_pattern`: **target/label**.
- `lab_result_received_two_days_later`: **future information / leakage** for an immediate prediction task.

The central principle is temporal and operational: **could the system really know this value at the moment it is expected to predict?**

</details>

# Part 3 - Build a self-contained smart-farm dataset

We now generate simple leaf-like images. These are **teaching data**, not a biological disease dataset and not a diagnostic system.

The generator creates three visible pattern classes:

- healthy-looking;
- spot-pattern;
- yellowing-pattern.

It also records a synthetic numerical variable, `affected_percent`, so that the **same collection of images** can illustrate classification and regression.

In [ ]:
# Core imports. These are available in a standard Google Colab environment.
import copy
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageEnhance
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, confusion_matrix, classification_report
)
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)

In [ ]:
CLASS_NAMES = ["healthy-looking", "spot-pattern", "yellowing-pattern"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
IMG_SIZE = 32


def make_leaf_image(label_name, rng, shifted=False):
    """Create one synthetic RGB leaf-like teaching image and its approximate affected percentage."""
    size = IMG_SIZE
    # dark neutral background
    bg = int(rng.integers(18, 34))
    im = Image.new("RGB", (size, size), (bg, bg, bg))
    draw = ImageDraw.Draw(im)

    # Leaf ellipse with small random movement/size variation
    margin_x = int(rng.integers(5, 8))
    margin_y = int(rng.integers(3, 6))
    bbox = [margin_x, margin_y, size-margin_x, size-margin_y]

    base_green = int(rng.integers(125, 190))
    leaf_color = (int(rng.integers(35, 75)), base_green, int(rng.integers(35, 75)))
    draw.ellipse(bbox, fill=leaf_color)

    # central vein
    draw.line((size//2, margin_y+2, size//2, size-margin_y-2), fill=(170, 205, 125), width=1)

    affected = 0.0

    if label_name == "spot-pattern":
        n_spots = int(rng.integers(4, 10))
        spot_area = 0
        for _ in range(n_spots):
            r = int(rng.integers(1, 3))
            x = int(rng.integers(margin_x+4, size-margin_x-4))
            y = int(rng.integers(margin_y+4, size-margin_y-4))
            draw.ellipse((x-r, y-r, x+r, y+r), fill=(95, 58, 28))
            spot_area += math.pi * r * r
        leaf_area = math.pi * ((size-2*margin_x)/2) * ((size-2*margin_y)/2)
        affected = float(np.clip(100*spot_area/max(leaf_area,1), 2, 38))

    elif label_name == "yellowing-pattern":
        # translucent-looking yellow patches approximated with solid overlays
        n_patches = int(rng.integers(3, 7))
        patch_area = 0
        for _ in range(n_patches):
            rx = int(rng.integers(2, 5))
            ry = int(rng.integers(2, 5))
            x = int(rng.integers(margin_x+5, size-margin_x-5))
            y = int(rng.integers(margin_y+5, size-margin_y-5))
            draw.ellipse((x-rx, y-ry, x+rx, y+ry), fill=(205, 190, 58))
            patch_area += math.pi * rx * ry
        leaf_area = math.pi * ((size-2*margin_x)/2) * ((size-2*margin_y)/2)
        affected = float(np.clip(100*patch_area/max(leaf_area,1), 8, 65))

    else:
        affected = float(rng.uniform(0.0, 3.0))

    # mild rotation and brightness variation
    angle = float(rng.uniform(-8, 8))
    im = im.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=(bg,bg,bg))
    brightness = float(rng.uniform(0.88, 1.12))
    im = ImageEnhance.Brightness(im).enhance(brightness)

    if shifted:
        # Simulate a simple distribution shift: darker lighting + stronger rotation/noise.
        im = ImageEnhance.Brightness(im).enhance(float(rng.uniform(0.58, 0.78)))
        im = im.rotate(float(rng.uniform(-18,18)), resample=Image.Resampling.BILINEAR, fillcolor=(bg,bg,bg))
        arr = np.asarray(im).astype(np.float32)
        arr += rng.normal(0, 9, size=arr.shape)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        im = Image.fromarray(arr)

    arr = np.asarray(im).astype(np.uint8)
    return arr, affected


def build_dataset(n_per_class=300, shifted=False, seed=SEED):
    rng = np.random.default_rng(seed)
    images, labels, rows = [], [], []
    case = 0
    for class_name in CLASS_NAMES:
        for _ in range(n_per_class):
            arr, affected = make_leaf_image(class_name, rng, shifted=shifted)
            images.append(arr)
            labels.append(CLASS_TO_ID[class_name])
            rows.append({
                "case_id": f"L{case:04d}",
                "visual_pattern": class_name,
                "affected_percent": affected,
                "mean_red": float(arr[:,:,0].mean()),
                "mean_green": float(arr[:,:,1].mean()),
                "mean_blue": float(arr[:,:,2].mean()),
                "brightness": float(arr.mean()),
            })
            case += 1
    return np.stack(images), np.array(labels, dtype=np.int64), pd.DataFrame(rows)

images, labels, metadata = build_dataset(n_per_class=300, shifted=False, seed=SEED)
shift_images, shift_labels, shift_metadata = build_dataset(n_per_class=80, shifted=True, seed=SEED+1)

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Shifted images:", shift_images.shape)
display(metadata.head())

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(11, 7))
for row, name in enumerate(CLASS_NAMES):
    idxs = np.where(labels == CLASS_TO_ID[name])[0][:5]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(images[idx])
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(name, loc='left', fontsize=11)
plt.suptitle("Synthetic teaching images - five examples per class", fontsize=14)
plt.tight_layout()
plt.show()

## What is one observation here?

One observation is **one image of one leaf-like object at one moment**.

For the main classification task:


a. input feature: the image;  
b. target: `visual_pattern`;  
c. target form: categorical;  
d. task: supervised multiclass classification.

We can write this abstractly as

$$
\mathbf{x} \;\longrightarrow\; f_\theta(\mathbf{x}) \;\longrightarrow\; \hat{y}
$$

where:

- $\mathbf{x}$ is an input image;
- $f_\theta$ is the learned model with parameters $\theta$;
- $\hat{y}$ is the predicted class.

# Part 4 - Classification, regression, and clustering using one dataset

The purpose of this section is not to build the strongest possible models. It is to make the **difference among the tasks concrete**.

## 4.1 Classification

Question:

> **Which predefined visual pattern is shown in the image?**

Target examples:

- `healthy-looking`
- `spot-pattern`
- `yellowing-pattern`

Because the required output is a **known category**, this is classification.

Later we will train a CNN for this task.

## 4.2 Regression

Change the question:

> **What percentage of the visible leaf area is affected?**

Now the target is a number, for example:

$$
y = 27.4\%
$$

A numerical target makes this a **regression** problem.

The same input domain can therefore produce a different task simply because the required output changed.

In [ ]:
# A deliberately simple regression example using image summary statistics.
reg_features = ["mean_red", "mean_green", "mean_blue", "brightness"]
X_reg = metadata[reg_features].to_numpy()
y_reg = metadata["affected_percent"].to_numpy()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=SEED
)
reg = LinearRegression().fit(X_train_r, y_train_r)
pred_r = reg.predict(X_test_r)
mae = mean_absolute_error(y_test_r, pred_r)
rmse = mean_squared_error(y_test_r, pred_r) ** 0.5
print(f"Regression demonstration - MAE: {mae:.2f} percentage points")
print(f"Regression demonstration - RMSE: {rmse:.2f} percentage points")

plt.figure(figsize=(6,5))
plt.scatter(y_test_r, pred_r, alpha=0.55)
lims = [min(y_test_r.min(), pred_r.min()), max(y_test_r.max(), pred_r.max())]
plt.plot(lims, lims, '--')
plt.xlabel("True affected percentage")
plt.ylabel("Predicted affected percentage")
plt.title("Regression: numerical target")
plt.show()

For regression, two common error measures are:

### Mean Absolute Error

$$
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
$$

It answers:

> On average, how large is the absolute numerical error?

### Root Mean Squared Error

$$
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}
$$

RMSE penalizes large errors more strongly because the errors are squared before averaging.

## 4.3 Clustering

Change the question again:

> **Ignore the supplied class labels. Can we discover groups of visually/statistically similar observations?**

This is clustering because there is **no target label supplied to the clustering algorithm**.

A cluster ID such as `Cluster 0` does **not automatically mean** `healthy-looking` or `spot-pattern`. Clusters are discovered from similarity; interpretation comes afterwards.

In [ ]:
cluster_features = ["mean_red", "mean_green", "mean_blue", "brightness"]
X_cluster = metadata[cluster_features].to_numpy()
X_scaled = StandardScaler().fit_transform(X_cluster)
km = KMeans(n_clusters=3, random_state=SEED, n_init=20)
cluster_id = km.fit_predict(X_scaled)

cluster_table = pd.crosstab(cluster_id, metadata["visual_pattern"], rownames=["cluster"])
display(cluster_table)

plt.figure(figsize=(7,5))
plt.scatter(metadata["mean_red"], metadata["mean_green"], c=cluster_id, alpha=0.55)
plt.xlabel("Mean red value")
plt.ylabel("Mean green value")
plt.title("Clustering: groups discovered without using class labels")
plt.show()

## Compare the three tasks carefully

| Property | Classification | Regression | Clustering |
|---|---|---|---|
| Known target during training? | Yes | Yes | No |
| Typical output | category | number | discovered group |
| Example here | visual pattern | affected % | image groups |
| Supervised? | Yes | Yes | No |

### A frequent mistake

**Wrong reasoning:** “There are three groups, therefore this is clustering.”

**Correct reasoning:** The number of groups does not decide the task. If the three categories are predefined target labels and the model learns to predict them, it is classification. If no labels are provided and groups are discovered from similarity, it is clustering.

# Part 5 - Where do AI, machine learning, deep learning, computer vision, and CNNs fit?

These terms are not competing answers. They describe different levels or aspects of a system.

For our main task:

- **Artificial Intelligence (AI):** broad field concerned with systems that perform tasks associated with intelligent behaviour;
- **Machine Learning (ML):** the system learns patterns from data rather than relying only on hand-written rules;
- **Deep Learning (DL):** the model uses a neural network with multiple learned layers;
- **Computer Vision (CV):** the information being interpreted is visual/image data;
- **Image Classification:** the required output is one image-level category;
- **CNN:** one model architecture suitable for learning spatial image features.

In [ ]:
# Nested concept diagram
fig, ax = plt.subplots(figsize=(9,6))
ax.axis('off')
rects = [
    (0.05,0.08,0.90,0.82,"Artificial Intelligence"),
    (0.14,0.17,0.72,0.64,"Machine Learning"),
    (0.24,0.26,0.52,0.46,"Deep Learning"),
]
for x,y,w,h,label in rects:
    patch = FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.02',facecolor='white',linewidth=2)
    ax.add_patch(patch)
    ax.text(x+0.02,y+h-0.06,label,fontsize=13,weight='bold',va='top')
ax.text(0.50,0.47,"CNN\nused for an image-classification task",ha='center',va='center',fontsize=12)
ax.text(0.50,0.13,"Computer vision describes the visual-data application area;\nit overlaps with these modeling levels rather than being merely another nested box.",ha='center',fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.title("Related terms answer different questions", fontsize=14)
plt.show()

## Vision task checkpoint

Suppose a road-camera image contains six vehicles.

Compare these desired outputs:

1. `heavy-traffic` vs `light-traffic` for the whole image → **classification**;
2. number of vehicles = `6` → **regression/count prediction**;
3. boxes around each car, bus, and motorcycle → **object detection**;
4. exact pixels belonging to road, vehicles, pedestrians, and sky → **segmentation**.

Again, **the image does not determine the task; the required output does**.

In [ ]:
# Visual demonstration: same image, different possible outputs.
idx = np.where(labels == CLASS_TO_ID["spot-pattern"])[0][0]
img = images[idx]

fig, axes = plt.subplots(1,3,figsize=(12,4))
axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title("Classification\nOne label for whole image")

axes[1].imshow(img); axes[1].axis('off'); axes[1].set_title("Detection idea\nLocations + labels")
axes[1].add_patch(plt.Rectangle((9,8),6,6,fill=False,linewidth=2))
axes[1].add_patch(plt.Rectangle((18,17),6,6,fill=False,linewidth=2))

# Simple illustrative segmentation mask based on brown-ish pixels; pedagogical only.
r,g,b = img[:,:,0],img[:,:,1],img[:,:,2]
mask = (r > g*0.9) & (g < 115)
axes[2].imshow(mask, cmap='gray'); axes[2].axis('off'); axes[2].set_title("Segmentation idea\nPixel-level mask")
plt.tight_layout(); plt.show()

# Part 6 - Training, validation, and test data

A model should not be judged only on examples it has already studied.

A useful analogy is:

- **training set** → exercises used to learn;
- **validation set** → practice examination used to tune choices;
- **test set** → final unseen examination used for the final estimate of performance.

The test set should not guide routine model tuning.

In [ ]:
# Stratified split so each subset keeps a similar class balance.
indices = np.arange(len(labels))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, random_state=SEED, stratify=labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED, stratify=labels[temp_idx]
)

for name, idxs in [("train",train_idx),("validation",val_idx),("test",test_idx)]:
    counts = np.bincount(labels[idxs], minlength=len(CLASS_NAMES))
    print(f"{name:10s}: n={len(idxs):3d}, counts={dict(zip(CLASS_NAMES, counts))}")

## Why accuracy can mislead

Suppose 95 of 100 transactions are legitimate and only 5 are fraudulent.

A useless model that predicts **legitimate every time** obtains:

$$
\mathrm{Accuracy}=\frac{95}{100}=95\%.
$$

That sounds high, but it finds **zero fraud cases**.

Therefore, evaluation must reflect the actual question. For individual classes, precision and recall are often informative.

# Part 7 - Why a CNN for images?

A digital image is ultimately an array of numbers.

An RGB image with height $H$ and width $W$ can be represented as

$$
\mathbf{X}\in\mathbb{R}^{H\times W\times 3}.
$$

The three channels usually represent red, green, and blue intensity.

Humans may immediately see an edge, patch, or spot. A neural network initially receives numerical pixel values and must learn useful patterns from them.

In [ ]:
# Look at a small pixel patch rather than the semantic object.
example = images[np.where(labels == CLASS_TO_ID["yellowing-pattern"])[0][0]]
fig, axes = plt.subplots(1,2,figsize=(9,4))
axes[0].imshow(example); axes[0].axis('off'); axes[0].set_title("What we see")
patch = example[12:17,12:17,1]
im = axes[1].imshow(patch)
axes[1].set_title("A 5×5 patch of green-channel numbers")
for i in range(patch.shape[0]):
    for j in range(patch.shape[1]):
        axes[1].text(j,i,str(int(patch[i,j])),ha='center',va='center',fontsize=8)
plt.tight_layout(); plt.show()

## Convolution - the main idea

A small learnable filter (kernel) moves across local regions of the image and computes responses.

For one 2-D input channel, a simplified convolution-like operation can be written as

$$
Y_{i,j}
=
\sum_{u=0}^{K_h-1}\sum_{v=0}^{K_w-1}
X_{i+u,j+v}\,W_{u,v}+b.
$$

Interpretation:

- $X$ = input values;
- $W$ = filter/kernel weights;
- $b$ = bias;
- $Y_{i,j}$ = response at one spatial location.

During learning, the useful filters are learned from data rather than manually chosen.

In [ ]:
# Manual convolution example on tiny numbers.
X_small = np.array([
    [1,1,1,0,0],
    [1,1,1,0,0],
    [1,1,1,0,0],
    [0,0,0,1,1],
    [0,0,0,1,1],
], dtype=float)
W_small = np.array([
    [1,0,-1],
    [1,0,-1],
    [1,0,-1],
], dtype=float)

out = np.zeros((3,3))
for i in range(3):
    for j in range(3):
        region = X_small[i:i+3,j:j+3]
        out[i,j] = np.sum(region * W_small)

print("Input:\n", X_small)
print("\nKernel:\n", W_small)
print("\nOutput feature map:\n", out)

## ReLU - introduce nonlinearity

A common activation function is the Rectified Linear Unit:

$$
\operatorname{ReLU}(z)=\max(0,z).
$$

Examples:

- $\operatorname{ReLU}(-3)=0$
- $\operatorname{ReLU}(2.5)=2.5$

Without nonlinear activations, stacking many linear operations would still collapse into a linear transformation and would greatly limit what the network can represent.

## Pooling - reduce spatial size

A common operation is max pooling. A $2\times2$ max-pool keeps the largest value in each local $2\times2$ region.

This reduces the spatial dimensions and retains strong local responses.

A simplified CNN flow is therefore:

$$
\text{image}
\rightarrow
\text{convolution}
\rightarrow
\text{ReLU}
\rightarrow
\text{pooling}
\rightarrow
\text{deeper features}
\rightarrow
\text{class scores}.
$$

In [ ]:
# Diagram of the conceptual CNN pipeline.
fig, ax = plt.subplots(figsize=(12,3.5))
ax.axis('off')
labels_flow = ["Image", "Convolution\nlocal patterns", "ReLU\nnonlinearity", "Pooling\nsmaller maps", "Deeper features", "3 class\nscores"]
xs = np.linspace(0.03,0.84,len(labels_flow))
for x, lab in zip(xs,labels_flow):
    box = FancyBboxPatch((x,0.38),0.13,0.28,boxstyle='round,pad=0.015',facecolor='white',linewidth=1.5)
    ax.add_patch(box); ax.text(x+0.065,0.52,lab,ha='center',va='center',fontsize=9.5)
for i in range(len(xs)-1):
    ax.add_patch(FancyArrowPatch((xs[i]+0.13,0.52),(xs[i+1],0.52),arrowstyle='->',mutation_scale=13))
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.title("A CNN as a sequence of transformations",fontsize=14)
plt.show()

## Output shapes - only the essential formula

For a convolution along one spatial dimension, if the input size is $N$, kernel size is $K$, padding is $P$, and stride is $S$, then

$$
N_{\text{out}}
=
\left\lfloor\frac{N+2P-K}{S}\right\rfloor+1.
$$

Example: with $N=32$, $K=3$, $P=1$, and $S=1$,

$$
N_{\text{out}}
=
\left\lfloor\frac{32+2(1)-3}{1}\right\rfloor+1
=32.
$$

A $2\times2$ pooling layer with stride 2 then approximately halves 32 to 16.

# Part 8 - Build a small CNN

We will use a deliberately small network so that the architecture remains readable.

Conceptual architecture:

```text
32×32 RGB image
      ↓
Conv(3→8) + ReLU
      ↓
2×2 MaxPool     → 16×16
      ↓
Conv(8→16) + ReLU
      ↓
2×2 MaxPool     → 8×8
      ↓
Global average pooling
      ↓
Linear layer
      ↓
3 class scores (logits)
```

The final three values are not yet probabilities; they are called **logits**.

In [ ]:
# Convert images to PyTorch tensors: [N,H,W,C] -> [N,C,H,W], scale 0..255 to 0..1.
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).permute(0,3,1,2) / 255.0

X_tensor = to_tensor(images)
y_tensor = torch.tensor(labels, dtype=torch.long)

train_ds = TensorDataset(X_tensor[train_idx], y_tensor[train_idx])
val_ds = TensorDataset(X_tensor[val_idx], y_tensor[val_idx])
test_ds = TensorDataset(X_tensor[test_idx], y_tensor[test_idx])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print("One batch image shape:", next(iter(train_loader))[0].shape)

In [ ]:
class SmallLeafCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Linear(16, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.classifier(x)

model = SmallLeafCNN(len(CLASS_NAMES))
print(model)

with torch.no_grad():
    sample_logits = model(torch.zeros(4,3,32,32))
print("Output shape for 4 images:", sample_logits.shape)

## From logits to probabilities

If the model produces logits $z_1,\ldots,z_K$, softmax converts them to probabilities:

$$
p_k
=
\frac{e^{z_k}}
{\sum_{j=1}^{K}e^{z_j}}.
$$

The probabilities sum to 1.

For one training example whose correct class is $y$, multiclass cross-entropy can be written conceptually as

$$
\mathcal{L}=-\log p_y.
$$

If the model gives a high probability to the correct class, the loss is small. If it gives a low probability to the correct class, the loss is larger.

## What training actually does

The repeated learning loop is:

1. send a batch of images through the model;
2. obtain predictions/logits;
3. compare predictions with the known targets using a loss function;
4. compute gradients of the loss with respect to model parameters;
5. let an optimizer update those parameters;
6. repeat over many batches.

Symbolically:

$$
\theta \leftarrow \theta - \eta\,\nabla_\theta \mathcal{L}(\theta)
$$

where $\eta$ is the learning rate.

In [ ]:
def evaluate_loader(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)
            total_loss += loss.item() * len(yb)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += len(yb)
    return total_loss/total, correct/total

# Reinitialize for a clean deterministic training run.
torch.manual_seed(SEED)
model = SmallLeafCNN(len(CLASS_NAMES))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

history = {"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}
EPOCHS = 8

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*len(yb)
        pred = logits.argmax(dim=1)
        correct += (pred==yb).sum().item()
        total += len(yb)
    train_loss = running_loss/total
    train_acc = correct/total
    val_loss, val_acc = evaluate_loader(model, val_loader, loss_fn)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch:2d}: train loss={train_loss:.4f}, train acc={train_acc:.3f}, val acc={val_acc:.3f}")

In [ ]:
epochs = np.arange(1,EPOCHS+1)
plt.figure(figsize=(7,4.5))
plt.plot(epochs, history["train_loss"], marker='o', label='train loss')
plt.plot(epochs, history["val_loss"], marker='o', label='validation loss')
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Learning curve - loss")
plt.legend()
plt.show()

plt.figure(figsize=(7,4.5))
plt.plot(epochs, history["train_acc"], marker='o', label='train accuracy')
plt.plot(epochs, history["val_acc"], marker='o', label='validation accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0,1.05)
plt.title("Learning curve - accuracy")
plt.legend()
plt.show()

# Part 9 - Evaluate the trained classifier

Evaluation should answer specific questions.

### Accuracy

> Overall, what fraction of predictions were correct?

$$
\mathrm{Accuracy}
=
\frac{\text{number correct}}{\text{number evaluated}}.
$$

### Precision for a class

> When the model predicted this class, how often was it correct?

$$
\mathrm{Precision}=\frac{TP}{TP+FP}.
$$

### Recall for a class

> Of all examples that truly belonged to this class, how many did the model find?

$$
\mathrm{Recall}=\frac{TP}{TP+FN}.
$$

### Confusion matrix

A confusion matrix shows **which classes are being confused with which others**, rather than reducing everything to one number.

In [ ]:
def predict_loader(model, loader):
    model.eval()
    all_y, all_pred, all_prob = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            pred = probs.argmax(dim=1)
            all_y.append(yb.numpy())
            all_pred.append(pred.numpy())
            all_prob.append(probs.numpy())
    return np.concatenate(all_y), np.concatenate(all_pred), np.concatenate(all_prob)

true_test, pred_test, prob_test = predict_loader(model, test_loader)
test_acc = accuracy_score(true_test, pred_test)
print(f"Test accuracy: {test_acc:.3f}")
print("\nClassification report:")
print(classification_report(true_test, pred_test, target_names=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(true_test, pred_test)
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm)
ax.set_xticks(range(3), CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(3), CLASS_NAMES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Test confusion matrix")
for i in range(3):
    for j in range(3):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center')
plt.tight_layout(); plt.show()

In [ ]:
# Show a selection of predictions with confidence.
fig, axes = plt.subplots(3,5,figsize=(11,7))
show_idx = test_idx[:15]
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(X_tensor[show_idx]),dim=1).numpy()
preds = probs.argmax(axis=1)

for ax, global_idx, pr, pv in zip(axes.ravel(), show_idx, preds, probs):
    ax.imshow(images[global_idx])
    ax.axis('off')
    truth = CLASS_NAMES[labels[global_idx]]
    pred_name = CLASS_NAMES[pr]
    conf = pv[pr]
    ax.set_title(f"T: {truth}\nP: {pred_name} ({conf:.2f})",fontsize=8)
plt.suptitle("Unseen test examples")
plt.tight_layout(); plt.show()

## Error analysis

A strong AI practitioner does not stop at one metric.

For a wrong prediction, ask:

1. What was the true class?
2. What did the model predict?
3. How confident was it?
4. Is the image ambiguous, noisy, unusually dark, or visually similar to another class?
5. Could the training data be missing important variation?
6. Is the label itself questionable?

Error analysis turns evaluation into a path toward improvement.

In [ ]:
wrong_local = np.where(true_test != pred_test)[0]
if len(wrong_local) == 0:
    print("No test errors occurred in this synthetic run. That can happen because the teaching patterns are deliberately distinct.")
else:
    n_show = min(8, len(wrong_local))
    fig, axes = plt.subplots(1,n_show,figsize=(2.5*n_show,3))
    axes = np.atleast_1d(axes)
    for ax, loc in zip(axes, wrong_local[:n_show]):
        global_idx = test_idx[loc]
        ax.imshow(images[global_idx]); ax.axis('off')
        ax.set_title(f"T:{CLASS_NAMES[true_test[loc]]}\nP:{CLASS_NAMES[pred_test[loc]]}")
    plt.tight_layout(); plt.show()

# Part 10 - Distribution shift: good test performance is not the whole story

Real deployment data may differ from the training data.

Examples:

- darker lighting;
- different cameras;
- rotation and framing changes;
- different farms or backgrounds;
- new disease stages not represented during training.

This is broadly related to **distribution shift**: the data encountered later do not follow exactly the same distribution as the training examples.

In [ ]:
fig, axes = plt.subplots(3,5,figsize=(11,7))
for row, name in enumerate(CLASS_NAMES):
    idxs = np.where(shift_labels == CLASS_TO_ID[name])[0][:5]
    for col, idx in enumerate(idxs):
        axes[row,col].imshow(shift_images[idx]); axes[row,col].axis('off')
        if col == 0:
            axes[row,col].set_title(name, loc='left', fontsize=10)
plt.suptitle("Shifted teaching images: darker, noisier, more rotated")
plt.tight_layout(); plt.show()

In [ ]:
shift_tensor = to_tensor(shift_images)
shift_ds = TensorDataset(shift_tensor, torch.tensor(shift_labels,dtype=torch.long))
shift_loader = DataLoader(shift_ds,batch_size=128,shuffle=False)
true_shift, pred_shift, prob_shift = predict_loader(model, shift_loader)
shift_acc = accuracy_score(true_shift,pred_shift)
print(f"Ordinary test accuracy: {test_acc:.3f}")
print(f"Shifted-set accuracy:  {shift_acc:.3f}")

## What is augmentation, and why might it help?

**Data augmentation** creates plausible variations of training images, such as modest rotations or brightness changes.

The purpose is not to create random distortion for its own sake. It is to teach the model that **reasonable variations should not change the class**.

For example, if a leaf remains a spot-pattern leaf when the camera is rotated by a few degrees, training on mild rotations can improve robustness.

### Caution
An augmentation is appropriate only if it preserves the meaning of the label. A transformation that changes the real-world class should not be used as a label-preserving augmentation.

In [ ]:
# Demonstrate several mild augmentations without introducing extra libraries.
base = Image.fromarray(images[np.where(labels==CLASS_TO_ID["spot-pattern"])[0][3]])
variants = [
    ("original", base),
    ("rotate +10°", base.rotate(10, resample=Image.Resampling.BILINEAR)),
    ("rotate -10°", base.rotate(-10, resample=Image.Resampling.BILINEAR)),
    ("brighter", ImageEnhance.Brightness(base).enhance(1.2)),
    ("darker", ImageEnhance.Brightness(base).enhance(0.8)),
]
fig, axes = plt.subplots(1,len(variants),figsize=(13,3))
for ax,(title,v) in zip(axes,variants):
    ax.imshow(v); ax.axis('off'); ax.set_title(title)
plt.suptitle("Label-preserving augmentation ideas")
plt.tight_layout(); plt.show()

# Part 11 - Model versus complete intelligent agent

A trained CNN can map

$$
\text{image}\rightarrow\text{class probabilities}.
$$

That does **not** automatically make it a complete autonomous agent.

An operational system might instead perform:

$$
\text{sense}
\rightarrow
\text{predict}
\rightarrow
\text{apply decision rule}
\rightarrow
\text{act}
\rightarrow
\text{observe feedback}.
$$

For example, a low-confidence prediction might trigger **human review** rather than an automatic recommendation.

In [ ]:
fig, ax = plt.subplots(figsize=(12,3.5))
ax.axis('off')
agent_labels = ["Sensor / image", "CNN model", "Decision rule", "Action", "Feedback"]
xs = np.linspace(0.04,0.82,len(agent_labels))
for x, lab in zip(xs,agent_labels):
    box = FancyBboxPatch((x,0.40),0.14,0.26,boxstyle='round,pad=0.015',facecolor='white',linewidth=1.5)
    ax.add_patch(box); ax.text(x+0.07,0.53,lab,ha='center',va='center')
for i in range(len(xs)-1):
    ax.add_patch(FancyArrowPatch((xs[i]+0.14,0.53),(xs[i+1],0.53),arrowstyle='->',mutation_scale=14))
ax.text(0.29,0.26,"The CNN is one component, not the whole decision system.",ha='center',fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.title("Predictive model inside a larger agent/system")
plt.show()

## PEAS for an agent

A classic way to describe an intelligent agent is **PEAS**:

- **P - Performance measure:** how success is judged;
- **E - Environment:** where the agent operates;
- **A - Actuators:** how the agent acts on the environment/user;
- **S - Sensors:** how it receives information.

For a smart-farm support agent, a defensible PEAS description could be:

| PEAS element | Example |
|---|---|
| Performance | useful alerts, low harmful false alarms, timely escalation, calibrated confidence |
| Environment | farm, leaves, lighting conditions, farmer workflow |
| Actuators | show message, request another photo, escalate to expert |
| Sensors | camera image, optional approved contextual sensors |

PEAS describes the **agent interaction**, while classification/regression/clustering describe **learning tasks**. These concepts answer different questions.

In [ ]:
# Example decision policy built around the model's probabilities.
def agent_decision(probabilities, threshold=0.75):
    pred_id = int(np.argmax(probabilities))
    confidence = float(probabilities[pred_id])
    if confidence < threshold:
        return {"action":"request human review / another image", "confidence":confidence}
    return {"action":f"report likely visual pattern: {CLASS_NAMES[pred_id]}", "confidence":confidence}

print(agent_decision(np.array([0.12,0.31,0.57])))
print(agent_decision(np.array([0.04,0.91,0.05])))

# Part 12 - Transfer laboratory: unfamiliar scenarios

This is the most important section.

For every scenario, complete the same reasoning canvas **before opening the answer**.

### Transfer canvas

1. What is the real-world goal?
2. What is one observation/prediction unit?
3. What information is available as input?
4. What should the system output?
5. Is there a known target during training?
6. What form does the output take?
7. Therefore, what kind of task is this?
8. Why?
9. Why is it not the most plausible alternative?

## Scenario 1 - Mobile-money fraud

A system receives information about one transaction: amount, time, account history, transaction channel, and recent activity. It must output either `fraud` or `legitimate`.

Write your reasoning before opening the answer.

In [ ]:
transfer_1 = {
    "unit":"", "features":"", "target":"", "output_form":"",
    "task":"", "why":"", "why_not_regression":"", "why_not_clustering":""
}
transfer_1

<details>
<summary><b>Reveal Scenario 1</b></summary>

**Task: binary classification.**

- unit: one transaction;
- features: information available about that transaction and relevant prior history;
- target: fraud vs legitimate;
- output form: one of two predefined categories.

It is **not regression** because the required answer is not a continuous number.  
It is **not clustering** because labelled target categories are defined and the model is asked to predict one of them.

</details>

## Scenario 2 - Campus electricity demand

Every hour, a system receives recent electricity use, time of day, day of week, and weather measurements. It must predict electricity demand for the next hour in kWh.

In [ ]:
transfer_2 = {"unit":"", "target":"", "task":"", "why":"", "why_not_classification":""}
transfer_2

<details>
<summary><b>Reveal Scenario 2</b></summary>

**Task: supervised regression.**

- unit: one forecasting time point;
- target: next-hour electricity demand;
- output form: numerical quantity in kWh.

It is not ordinary classification because the system is not choosing from predefined discrete categories such as `low`, `medium`, and `high`.

</details>

## Scenario 3 - Customer behaviour without labels

A telecom company has usage statistics for many customers but no predefined customer-type labels. Analysts want the algorithm to discover groups of customers with similar behaviour.

In [ ]:
transfer_3 = {"unit":"", "target_exists":"", "task":"", "why":"", "why_not_classification":""}
transfer_3

<details>
<summary><b>Reveal Scenario 3</b></summary>

**Task: clustering.**

The goal is to discover structure in unlabeled data. There is no supplied target such as `type A`, `type B`, or `type C` for the algorithm to learn.

It is not classification merely because the result may contain several groups. Classification requires predefined target classes for supervised learning.

</details>

## Scenario 4 - Road-camera system

A camera image may contain many vehicles. The system must return a bounding box and class label for every visible car, bus, and motorcycle.

In [ ]:
transfer_4 = {"unit":"", "required_output":"", "task":"", "why_not_image_classification":""}
transfer_4

<details>
<summary><b>Reveal Scenario 4</b></summary>

**Task: object detection.**

The output requires both:

- **what** each object is; and
- **where** it is located.

Ordinary image classification would normally produce one label for the whole image and would not provide separate object locations.

</details>

## Scenario 5 - Student records, three different questions

For the same collection of student records, identify the task in each case:

A. Predict whether the student will pass or fail.  
B. Predict the final numerical mark.  
C. Without pass/fail or other group labels, discover common learning-behaviour profiles.

In [ ]:
transfer_5 = {"A":"", "B":"", "C":"", "main_lesson":""}
transfer_5

<details>
<summary><b>Reveal Scenario 5</b></summary>

A. **Classification** - target is a predefined category: pass/fail.  
B. **Regression** - target is a numerical mark.  
C. **Clustering** - groups are discovered without predefined target labels.

### Main lesson
The dataset or domain does not uniquely determine the ML task. **The question being asked determines the target and therefore the task.**

</details>

# Part 13 - Final competence check

You should now be able to answer these questions without copying a definition.

### Problem framing
- Can I identify one observation/prediction unit?
- Can I separate candidate features from the target?
- Can I detect obvious future-information leakage?
- Can I decide whether the required output is a category, number, discovered group, object location, pixel mask, or action?

### Task reasoning
- Can I explain **why** a problem is classification rather than merely name it?
- Can I explain the difference between classification and clustering even if both produce groups?
- Can I explain how the same dataset can support different tasks when the question changes?

### CNN understanding
- Can I explain why images are arrays of numbers?
- Can I explain what convolution, ReLU, and pooling contribute?
- Can I explain what logits and softmax probabilities are?
- Can I describe training as prediction → loss → gradients → parameter update?

### Evaluation
- Can I explain why unseen test data matter?
- Can I interpret accuracy, precision, recall, and a confusion matrix?
- Can I explain why distribution shift can reduce deployment performance?

### Systems thinking
- Can I distinguish a predictive model from a complete agent?
- Can I give a defensible PEAS description for an agent?

If any answer is “not yet,” return to the relevant section and explain it aloud in your own words.

# Optional extension challenges

These are deliberately placed at the end so they do not obstruct the core AI Basics path.

1. Add a fourth visual class to the synthetic image generator. What else must change?
2. Train a regression neural network to estimate `affected_percent` directly from images.
3. Add mild training-time augmentation and compare ordinary versus shifted-set performance.
4. Create an intentionally imbalanced classification dataset and compare accuracy with per-class recall.
5. Inspect intermediate CNN feature maps and describe what changed from early to later layers.
6. Replace the synthetic images with a properly documented real dataset and repeat the entire framing process before training.